In [21]:
#==========================================================================
# IRB-Style PD (Proxy) + Policy + Portfolio EL Workflow + Model Governance
# =========================================================================
"""
Dataset:
- Target = Personal.loan (0=not approved, 1=approved)
- Since default outcome is NOT available, we build a PD proxy:
      prob_def = 1 - P(approved)

Outputs:
- prob_def, loan_approval (policy), loan_amount_approved, EAD, LGD, EL
- model artifact (.joblib) and scored file (.csv)

Author notes:
- This is structured like an IRB workflow: data treatment -> dev/val/test -> calibration -> policy layer -> portfolio metrics
"""
    
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

In [22]:
# -----------------------------
# 1) Load data
# -----------------------------
DATA_PATH = "Bank_Loan_Approval_copy.csv"  # <- for GitHub repo, place csv under /data
# If running locally from this notebook environment, you can temporarily set:
# DATA_PATH = "/mnt/data/Bank_Loan_Approval_copy.csv"

df = pd.read_csv(DATA_PATH)

In [23]:
# -----------------------------
# 2) Standardize column names
# -----------------------------
df.columns = [c.strip().replace(" ", "_").replace(".", "_") for c in df.columns]
df.rename(columns={"Avg_CC_Score": "Avg_CC_score", "personal_loan": "Personal_loan"}, inplace=True)

TARGET = "Personal_loan"

# Drop identifiers / high-cardinality fields (ZIP often creates leakage/noise unless engineered)
DROP_COLS = ["ID", "ZIP_Code"]

X = df.drop(columns=[TARGET] + [c for c in DROP_COLS if c in df.columns])
y = df[TARGET].astype(int)

In [24]:
# -----------------------------
# 3) Dev / Val / Test split (60/20/20)
# -----------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [25]:
# -----------------------------
# 4) Outlier treatment helper (Winsorization)
# -----------------------------
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Caps numeric outliers using training quantiles (IRB-style: fit on DEV only).
    """
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
        self.bounds_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            lo = X[c].quantile(self.lower)
            hi = X[c].quantile(self.upper)
            self.bounds_[c] = (lo, hi)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c, (lo, hi) in self.bounds_.items():
            X[c] = X[c].clip(lo, hi)
        return X

In [26]:
# -----------------------------
# 5) Preprocess + Model (Logit) + Calibration
# -----------------------------
# Treat Education as ordinal numeric (1–3). Dummies pass through.
numeric_features = ["Age", "Experience", "Income", "Family", "Avg_CC_score", "Mortgage", "Education"]
binary_features = ["Securities_Account", "CD_Account", "Online", "CreditCard"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # missing numeric -> median
    ("winsor", Winsorizer(0.01, 0.99)),              # cap outliers based on DEV
    ("scaler", StandardScaler())                     # scale for stable logit coefficients
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))  # missing dummies -> mode
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("bin", binary_transformer, binary_features),
    ],
    remainder="drop"
)

base_logit = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    class_weight="balanced"  # helps when approvals are minority (~10%)
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("logit", base_logit)
])

# Calibration improves probability quality (important for PD-style use)
calibrated = CalibratedClassifierCV(estimator=model, method="sigmoid", cv=5)
calibrated.fit(X_train, y_train)

,estimator,Pipeline(step..._iter=2000))])
,method,'sigmoid'
,cv,5
,n_jobs,None
,ensemble,'auto'
,transformers,"[('num', ...), ('bin', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


In [27]:
# -----------------------------
# 6) Evaluation (Dev/Val/Test)
# -----------------------------
def eval_model(clf, X, y, name):
    p = clf.predict_proba(X)[:, 1]
    return {
        "dataset": name,
        "AUC": roc_auc_score(y, p),
        "Brier": brier_score_loss(y, p),
        "predicted_approval_rate_at_0_5": float((p >= 0.5).mean()),
        "actual_approval_rate": float(y.mean())
    }

metrics = pd.DataFrame([
    eval_model(calibrated, X_train, y_train, "train"),
    eval_model(calibrated, X_val, y_val, "val"),
    eval_model(calibrated, X_test, y_test, "test"),
])

print("\nMODEL QUALITY SUMMARY")
print(metrics.to_string(index=False))


MODEL QUALITY SUMMARY
dataset      AUC    Brier  predicted_approval_rate_at_0_5  actual_approval_rate
  train 0.957245 0.041421                        0.068333                 0.096
    val 0.960626 0.040287                        0.074000                 0.096
   test 0.973728 0.034044                        0.070000                 0.096


In [28]:
# -----------------------------
# 7) PD proxy + Credit policy
# -----------------------------
# PD proxy (because target is approval, not default)
p_approve_all = calibrated.predict_proba(X)[:, 1]
prob_def = 1.0 - p_approve_all

"""
Policy cutoff (PD criterion):
- In real credit policy, you pick PD cutoffs based on objectives (approval rate, risk appetite, profit).
- Here we choose a cutoff that roughly matches the historical approval rate in this dataset (~9–10%),
  *using the validation set* as the policy tuning set.

This keeps the policy “portfolio-consistent” with what the dataset represents.
"""

# Find PD cutoff on validation that matches historical approval rate (proxy approach)
p_val = calibrated.predict_proba(X_val)[:, 1]
pd_val = 1.0 - p_val
target_approval_rate = float(y.mean())

grid = np.linspace(0.05, 0.95, 181)
best_cut = None
best_gap = 10.0

for c in grid:
    appr = float((pd_val <= c).mean())
    gap = abs(appr - target_approval_rate)
    if gap < best_gap:
        best_gap = gap
        best_cut = c

PD_CUTOFF = float(best_cut)
print(f"\nSelected PD_CUTOFF (policy) = {PD_CUTOFF:.3f} (matches historical approval rate ~{target_approval_rate:.3f})")

loan_approval = (prob_def <= PD_CUTOFF).astype(int)


Selected PD_CUTOFF (policy) = 0.650 (matches historical approval rate ~0.096)


In [29]:
# -----------------------------
# 8) Assign loan amount + EAD/LGD/EL (portfolio level)
# -----------------------------
"""
Loan amount:
- Randomly assigned ONLY for approved applicants (per your request).
- Between $5,000 and $45,000 inclusive.

EAD:
- For simplicity, treat as term loan with CCF=100% => EAD = approved amount

LGD:
- With no collateral/recoveries fields, we use a heuristic IRB-like proxy:
  - Base LGD decreases as Avg_CC_score improves (lower loss severity)
  - Securities/CD accounts slightly reduce LGD (proxy for liquidity/collateral buffer)
  - Mortgage>0 slightly increases LGD (proxy for debt burden)
  - Clamp to a realistic range [0.25, 0.65]
"""

rng = np.random.default_rng(42)

loan_amount_approved = np.where(
    loan_approval == 1,
    rng.integers(5000, 45001, size=len(df)),
    0
)

EAD = loan_amount_approved.astype(float)  # CCF = 1.0

avg_cc = df["Avg_CC_score"].astype(float).values
LGD = 0.55 - 0.02 * avg_cc
LGD = LGD - 0.05 * df["Securities_Account"].values - 0.05 * df["CD_Account"].values
LGD = LGD + 0.03 * (df["Mortgage"].values > 0).astype(int)
LGD = np.clip(LGD, 0.25, 0.65)

EL = prob_def * LGD * EAD

In [30]:
# -----------------------------
# 9) Final scored output
# -----------------------------
scored = df.copy()
scored["prob_def"] = prob_def
scored["loan_approval"] = loan_approval
scored["loan_amount_approved"] = loan_amount_approved
scored["EAD"] = EAD
scored["LGD"] = LGD
scored["EL"] = EL

# Portfolio-level metrics (corporate portfolio aggregation)
portfolio = {
    "n_applicants": int(len(scored)),
    "approved_count": int(scored["loan_approval"].sum()),
    "approval_rate": float(scored["loan_approval"].mean()),
    "total_EAD": float(scored["EAD"].sum()),
    "avg_LGD_approved": float(scored.loc[scored["loan_approval"] == 1, "LGD"].mean()) if scored["loan_approval"].sum() else np.nan,
    "portfolio_EL": float(scored["EL"].sum()),
    "EL_rate_on_EAD": float(scored["EL"].sum() / scored["EAD"].sum()) if scored["EAD"].sum() else np.nan
}

print("\nPORTFOLIO SUMMARY (EAD/LGD/EL)")
for k, v in portfolio.items():
    print(f"{k}: {v}")


PORTFOLIO SUMMARY (EAD/LGD/EL)
n_applicants: 5000
approved_count: 467
approval_rate: 0.0934
total_EAD: 11714630.0
avg_LGD_approved: 0.44522612419700214
portfolio_EL: 1666947.8959918444
EL_rate_on_EAD: 0.14229624802420943


In [31]:
# -----------------------------
# 10) Export artifacts
# -----------------------------
os.makedirs("artifacts", exist_ok=True)

MODEL_OUT = "artifacts/irb_pd_proxy_model_calibrated.joblib"
SCORED_OUT = "artifacts/bank_loan_scored_irb.csv"

joblib.dump(calibrated, MODEL_OUT)
scored.to_csv(SCORED_OUT, index=False)

print(f"\nSaved model to:  {MODEL_OUT}")
print(f"Saved scored to: {SCORED_OUT}")


Saved model to:  artifacts/irb_pd_proxy_model_calibrated.joblib
Saved scored to: artifacts/bank_loan_scored_irb.csv


In [ ]:
# ===========================================================================
# Model Governance & Monitoring Section
#============================================================================
""" This section ensures the model meets risk management, regulatory, and production monitoring standards.
Governance includes: Model performance monitoring, Stability monitoring (PSI), Backtesting, Challenger model comparison, Cutoff policy validation,
Portfolio risk monitoring, Model documentation
"""

In [32]:
#===================================================
# 1-- Population Stability Index (PSI)
#===================================================
""" PSI measures whether data distribution has shifted between development and new data.
| PSI      | Meaning                             |
| -------- | ----------------------------------- |
| <0.1     | Stable population                   |
| 0.1–0.25 | Moderate shift                      |
| >0.25    | Major shift (model review required) |
"""
def calculate_psi(expected, actual, bins=10):

    expected = pd.Series(expected)
    actual = pd.Series(actual)

    quantiles = np.linspace(0,1,bins+1)

    breakpoints = expected.quantile(quantiles)

    expected_counts = np.histogram(expected, breakpoints)[0]
    actual_counts = np.histogram(actual, breakpoints)[0]

    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)

    psi = np.sum((actual_perc - expected_perc) * np.log((actual_perc + 1e-6)/(expected_perc + 1e-6)))

    return psi

# Predict approval probabilities
p_train = calibrated.predict_proba(X_train)[:,1]
p_test  = calibrated.predict_proba(X_test)[:,1]

# Convert to PD proxy
pd_train = 1 - p_train
pd_test  = 1 - p_test

psi_pd = calculate_psi(pd_train, pd_test)

print("PSI (PD distribution):", psi_pd)

PSI (PD distribution): 0.022356951092600986


In [33]:
#==================================================
# 2 -- Model Backtesting
#==================================================
"""Backtesting checks whether predicted risk aligns with actual outcomes. 
Ensures the model rank orders risk correctly.
Higher predicted probability corresponds to higher actual outcome
"""
backtest = pd.DataFrame({
    "prob_approve": p_test,
    "actual": y_test
})

backtest["decile"] = pd.qcut(backtest["prob_approve"], 10)

summary = backtest.groupby("decile", observed=False).agg(
    predicted_rate=("prob_approve","mean"),
    actual_rate=("actual","mean"),
    count=("actual","count")
)

print(summary)

                      predicted_rate  actual_rate  count
decile                                                  
(-0.000894, 0.00121]        0.000672         0.00    100
(0.00121, 0.00235]          0.001782         0.00    100
(0.00235, 0.00458]          0.003432         0.00    100
(0.00458, 0.00739]          0.005790         0.00    100
(0.00739, 0.012]            0.009333         0.00    100
(0.012, 0.0231]             0.017371         0.01    100
(0.0231, 0.0461]            0.033515         0.01    100
(0.0461, 0.0947]            0.066787         0.05    100
(0.0947, 0.299]             0.171903         0.17    100
(0.299, 0.993]              0.674674         0.72    100


In [34]:
#============================================
# 3 -- Challenger Model; Random Forest
#===========================================
""" Banks never rely on one model only. They maintain challenger models to compare performance.
Typical challengers:
| Model               | Reason                  |
| ------------------- | ----------------------- |
| Logistic Regression | Basel standard          |
| Random Forest       | nonlinear interactions  |
| Gradient Boosting   | higher predictive power |
"""
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict_proba(X_test)[:,1]

rf_auc = roc_auc_score(y_test, rf_pred)

print("Random Forest AUC:", rf_auc)

Random Forest AUC: 0.9978221792035398


In [35]:
#==================================================
# 4 --- Cutoff Sensitivity Analysis
#==================================================
""" Analysis of approval vs risk tradeoff. Risk managers choose cutoff based on: approval rate targets, expected loss, capital allocation.
"""

cutoffs = np.arange(0.05,0.5,0.05)

results = []

for c in cutoffs:

    approve = (prob_def <= c)

    approval_rate = approve.mean()

    bad_rate = prob_def[approve].mean()

    results.append([c, approval_rate, bad_rate])

cutoff_table = pd.DataFrame(results,
                            columns=["PD_cutoff","approval_rate","avg_PD"])

print(cutoff_table)

   PD_cutoff  approval_rate    avg_PD
0       0.05         0.0148  0.025949
1       0.10         0.0232  0.042125
2       0.15         0.0296  0.059329
3       0.20         0.0356  0.078243
4       0.25         0.0412  0.098688
5       0.30         0.0472  0.121221
6       0.35         0.0514  0.137133
7       0.40         0.0570  0.160715
8       0.45         0.0628  0.184811


In [36]:
#==================================================
# 5 - Portfolio Expected Loss Monitoring
#==================================================
""" Portfolio metrics must be monitored periodically.
"""

portfolio_metrics = {

"Total_EAD": EAD.sum(),

"Average_PD": prob_def.mean(),

"Average_LGD": LGD.mean(),

"Expected_Loss": EL.sum(),

"EL_rate": EL.sum()/EAD.sum()

}

print(portfolio_metrics)

{'Total_EAD': np.float64(11714630.0), 'Average_PD': np.float64(0.9035022412864404), 'Average_LGD': np.float64(0.51222924), 'Expected_Loss': np.float64(1666947.8959918444), 'EL_rate': np.float64(0.14229624802420943)}
